# FlagOS Track 1 — log10 baseline (university entry)

Minimal runnable kernel that emits the leaderboard `submission.csv`.
Track-1 panel work (full operator repo + benchmarks) lives at the linked GitHub repo.


## Environment

In [ ]:
import math, os
import numpy as np
import pandas as pd
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
try:
    import triton, triton.language as tl
    HAS_TRITON = True
    print('triton:', triton.__version__)
except Exception as e:
    HAS_TRITON = False
    print('triton: not available ->', e)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_TRITON = HAS_TRITON and DEVICE == 'cuda'


## Triton `log10` kernel + `torch.log10` fallback

Identity: `log10(x) = ln(x) * 0.4342944819032518`. fp16/bf16 promoted to fp32 inside the kernel.

In [ ]:
RECIP_LN10 = 1.0 / math.log(10.0)

if HAS_TRITON:
    @triton.autotune(
        configs=[triton.Config({'BLOCK_SIZE': bs}, num_warps=nw, num_stages=ns)
                 for bs in (1024, 2048, 4096, 8192) for nw in (4, 8) for ns in (2, 3)],
        key=['n_elements'],
    )
    @triton.jit
    def _log10_kernel(x_ptr, y_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
        pid = tl.program_id(0)
        offs = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
        mask = offs < n_elements
        x = tl.load(x_ptr + offs, mask=mask, other=1.0)
        y = (tl.log(x.to(tl.float32)) * 0.4342944819032518).to(x.dtype)
        tl.store(y_ptr + offs, y, mask=mask)

def log10(x: torch.Tensor) -> torch.Tensor:
    if not (USE_TRITON and x.is_cuda and x.dtype in (torch.float16, torch.bfloat16, torch.float32)):
        return torch.log10(x)
    if not x.is_contiguous():
        x = x.contiguous()
    out = torch.empty_like(x)
    n = x.numel()
    if n == 0:
        return out
    grid = lambda meta: (triton.cdiv(n, meta['BLOCK_SIZE']),)
    _log10_kernel[grid](x, out, n)
    return out


## Correctness vs `torch.log10`

In [ ]:
TOL = {torch.float16: (1e-3, 1e-3), torch.bfloat16: (1e-2, 1.6e-2), torch.float32: (1e-5, 1.3e-6)}
for dtype, (rtol, atol) in TOL.items():
    torch.manual_seed(0)
    x = torch.rand(1024, 1024, device=DEVICE, dtype=dtype) + 0.1
    torch.testing.assert_close(log10(x), torch.log10(x), equal_nan=True, rtol=rtol, atol=atol)
    print(f'ok  random {str(dtype):>16}')
edge = torch.tensor([0., -1., 1., 10., 1e-30, 1e30, float('inf'), -float('inf'), float('nan')],
                    device=DEVICE, dtype=torch.float32)
torch.testing.assert_close(log10(edge), torch.log10(edge), equal_nan=True, rtol=1e-5, atol=1.3e-6)
print('ok  edge values:', [round(v, 4) for v in log10(edge).cpu().tolist()])


## Emit `submission.csv`

In [ ]:
NUM_ROWS = 1000
x = np.linspace(0.001, 1000.0, NUM_ROWS, dtype=np.float64)
target = np.log10(x)
submission = pd.DataFrame({'ID': np.arange(NUM_ROWS, dtype=np.int64), 'target': target})
submission.to_csv('submission.csv', index=False)
print('wrote submission.csv rows=%d' % len(submission))
print(submission.head(3))
print('...')
print(submission.tail(3))
